In [0]:
%run ./1_config

nyc_taxi


catalog : nyc_taxi
schemas : landing / bronze / silver / gold
landing volume : /Volumes/nyc_taxi/landing/files


In [0]:
from pyspark.sql import functions as F, Window
from delta.tables import DeltaTable

spark.sql(f"USE CATALOG {CATALOG}")

DataFrame[]

In [0]:
DQ

{'min_duration_min': 1,
 'max_duration_min': 720,
 'min_trip_distance': 0.1,
 'max_trip_distance': 100,
 'max_passengers': 6,
 'min_total_amount': 0,
 'max_total_amount': 1000,
 'valid_from_date': '2024-01-01',
 'valid_to_date': '2027-12-31'}

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {T_SILVER_TRIPS} (
  trip_key            STRING  NOT NULL COMMENT 'sha2 business key, dedup + merge key',
  vendor_id           INT,
  pickup_ts           TIMESTAMP,
  dropoff_ts          TIMESTAMP,
  pickup_date         DATE,
  pickup_hour         INT,
  passenger_count     INT,
  trip_distance       DOUBLE,
  trip_duration_min   DOUBLE,
  avg_speed_mph       DOUBLE,
  rate_code_id        INT,
  store_and_fwd_flag  STRING,
  pu_location_id      INT,
  do_location_id      INT,
  payment_type_id     INT,
  fare_amount         DOUBLE,
  extra               DOUBLE,
  mta_tax             DOUBLE,
  tip_amount          DOUBLE,
  tolls_amount        DOUBLE,
  improvement_surcharge DOUBLE,
  congestion_surcharge  DOUBLE,
  airport_fee         DOUBLE,
  total_amount        DOUBLE,
  tip_pct             DOUBLE,
  is_airport_trip     BOOLEAN,
  _source_file        STRING,
  _ingested_at        TIMESTAMP,
  _processed_at       TIMESTAMP
)
USING DELTA
COMMENT 'Cleansed, deduplicated yellow taxi trips'
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {T_SILVER_QUARANTINE} (
  trip_key STRING, pickup_ts TIMESTAMP, dropoff_ts TIMESTAMP, passenger_count INT,
  trip_distance DOUBLE, fare_amount DOUBLE, total_amount DOUBLE,
  pu_location_id INT, do_location_id INT,
  dq_failures ARRAY<STRING>, _source_file STRING, _quarantined_at TIMESTAMP
) USING DELTA
COMMENT 'Rows rejected by silver DQ rules, kept for investigation'
""")

DataFrame[]

## Transformations + DQ rules

In [0]:
def conform(df):
    """Rename to snake_case, cast, and derive the columns downstream needs."""
    return (df
        .withColumn("pickup_ts",  F.col("tpep_pickup_datetime").cast("timestamp"))
        .withColumn("dropoff_ts", F.col("tpep_dropoff_datetime").cast("timestamp"))
        .withColumn("trip_key", F.sha2(F.concat_ws("||",
            F.coalesce(F.col("VendorID").cast("string"), F.lit("")),
            F.coalesce(F.col("tpep_pickup_datetime").cast("string"), F.lit("")),
            F.coalesce(F.col("tpep_dropoff_datetime").cast("string"), F.lit("")),
            F.coalesce(F.col("PULocationID").cast("string"), F.lit("")),
            F.coalesce(F.col("DOLocationID").cast("string"), F.lit("")),
            F.coalesce(F.col("trip_distance").cast("string"), F.lit("")),
            F.coalesce(F.col("total_amount").cast("string"), F.lit("")),
        ), 256))
        .withColumn("pickup_date", F.to_date("pickup_ts"))
        .withColumn("pickup_hour", F.hour("pickup_ts"))
        .withColumn("trip_duration_min",
            F.round((F.col("dropoff_ts").cast("long") - F.col("pickup_ts").cast("long")) / 60.0, 2))
        .withColumn("avg_speed_mph",
            F.when(F.col("trip_duration_min") > 0,
                   F.round(F.col("trip_distance") / (F.col("trip_duration_min") / 60.0), 2)))
        .withColumn("vendor_id",       F.col("VendorID").cast("int"))
        .withColumn("rate_code_id",    F.coalesce(F.col("RatecodeID").cast("int"), F.lit(99)))
        .withColumn("payment_type_id", F.coalesce(F.col("payment_type").cast("int"), F.lit(5)))
        .withColumn("pu_location_id",  F.col("PULocationID").cast("int"))
        .withColumn("do_location_id",  F.col("DOLocationID").cast("int"))
        .withColumn("passenger_count", F.col("passenger_count").cast("int"))
        .withColumn("store_and_fwd_flag", F.upper(F.trim(F.coalesce(F.col("store_and_fwd_flag"), F.lit("N")))))
        .withColumn("airport_fee",          F.coalesce(F.col("airport_fee").cast("double"), F.lit(0.0)))
        .withColumn("congestion_surcharge", F.coalesce(F.col("congestion_surcharge").cast("double"), F.lit(0.0)))
        .withColumn("tip_pct",
            F.when(F.col("fare_amount") > 0,
                   F.round(F.col("tip_amount") / F.col("fare_amount") * 100, 2)).otherwise(F.lit(0.0)))
        .withColumn("is_airport_trip",
            F.col("PULocationID").isin(1, 132, 138) | F.col("DOLocationID").isin(1, 132, 138))
    )


def add_dq_failures(df):
    """One array column listing every rule a row broke. Empty array = clean."""
    rules = [
        (F.col("pickup_ts").isNull() | F.col("dropoff_ts").isNull(),            "null_timestamp"),
        (F.col("pu_location_id").isNull() | F.col("do_location_id").isNull(),   "null_location"),
        (F.col("dropoff_ts") <= F.col("pickup_ts"),                             "dropoff_before_pickup"),
        (F.col("trip_duration_min") < DQ["min_duration_min"],                   "duration_too_short"),
        (F.col("trip_duration_min") > DQ["max_duration_min"],                   "duration_too_long"),
        (F.col("trip_distance") < DQ["min_trip_distance"],                      "distance_too_short"),
        (F.col("trip_distance") > DQ["max_trip_distance"],                      "distance_too_long"),
        (F.col("passenger_count").isNull() | (F.col("passenger_count") < 1),    "invalid_passenger_count"),
        (F.col("passenger_count") > DQ["max_passengers"],                       "too_many_passengers"),
        (F.col("fare_amount").isNull() | (F.col("fare_amount") < 0), "invalid_fare"),
        (F.col("total_amount").isNull() | (F.col("total_amount") <= DQ["min_total_amount"]), "non_positive_total"),
        (F.col("total_amount") > DQ["max_total_amount"],                        "total_out_of_range"),
        (F.col("pickup_date") < F.lit(DQ["valid_from_date"]).cast("date"),      "pickup_date_too_old"),
        (F.col("pickup_date") > F.lit(DQ["valid_to_date"]).cast("date"),        "pickup_date_in_future"),
        (F.col("avg_speed_mph") > 90,                                           "implausible_speed"),
    ]
    arr = F.array(*[F.when(cond, F.lit(name)) for cond, name in rules])
    return df.withColumn("dq_failures", F.array_compact(arr))


SILVER_COLS = ["trip_key","vendor_id","pickup_ts","dropoff_ts","pickup_date","pickup_hour",
    "passenger_count","trip_distance","trip_duration_min","avg_speed_mph","rate_code_id",
    "store_and_fwd_flag","pu_location_id","do_location_id","payment_type_id","fare_amount",
    "extra","mta_tax","tip_amount","tolls_amount","improvement_surcharge","congestion_surcharge",
    "airport_fee","total_amount","tip_pct","is_airport_trip","_source_file","_ingested_at"]

## foreachBatch: quarantine + MERGE

In [0]:
def process_batch(batch_df, batch_id):
    if batch_df.isEmpty():
        return

    # Bronze ka _ingest_time column Silver mein _ingested_at ban jata hai
    batch_df = batch_df.withColumn("_ingested_at", F.col("_ingest_time"))

    scored = add_dq_failures(conform(batch_df))

    # --- rejects -> quarantine ---
    (scored.filter(F.size("dq_failures") > 0)
        .select("trip_key","pickup_ts","dropoff_ts","passenger_count","trip_distance",
                "fare_amount","total_amount","pu_location_id","do_location_id",
                "dq_failures","_source_file",
                F.current_timestamp().alias("_quarantined_at"))
        .write.mode("append").saveAsTable(T_SILVER_QUARANTINE))

    # --- clean rows: batch ke andar dedup (latest ingest wins), phir MERGE ---
    w = Window.partitionBy("trip_key").orderBy(F.col("_ingested_at").desc())
    clean = (scored.filter(F.size("dq_failures") == 0)
        .withColumn("_rn", F.row_number().over(w)).filter("_rn = 1").drop("_rn")
        .select(*SILVER_COLS)
        .withColumn("_processed_at", F.current_timestamp()))

    (DeltaTable.forName(spark, T_SILVER_TRIPS).alias("t")
        .merge(clean.alias("s"), "t.trip_key = s.trip_key")
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute())

In [0]:
q = (spark.readStream
        .table(T_BRONZE_TRIPS)
        .writeStream
        .foreachBatch(process_batch)
        .option("checkpointLocation", CKPT)
        .trigger(availableNow=True)
        .start())
q.awaitTermination()
print("silver load finished")

silver load finished


## Zone staging 

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {T_SILVER_ZONE_STG} AS
SELECT
  CAST(LocationID AS INT)                       AS location_id,
  initcap(trim(Borough))                        AS borough,
  trim(Zone)                                    AS zone_name,
  trim(service_zone)                            AS service_zone,
  sha2(concat_ws('||', trim(Borough), trim(Zone), trim(service_zone)), 256) AS hash_diff,
  current_timestamp()                           AS _staged_at
FROM {T_BRONZE_ZONES}
WHERE LocationID IS NOT NULL
QUALIFY row_number() OVER (PARTITION BY LocationID ORDER BY _ingest_time DESC) = 1
""")

display(spark.sql(f"SELECT count(*) AS zones FROM {T_SILVER_ZONE_STG}"))

zones
265


## Verify

In [0]:
display(spark.sql(f"""
SELECT (SELECT count(*) FROM {T_BRONZE_TRIPS})      AS bronze_rows,
       (SELECT count(*) FROM {T_SILVER_TRIPS})      AS silver_rows,
       (SELECT count(*) FROM {T_SILVER_QUARANTINE}) AS quarantined_rows
"""))

bronze_rows,silver_rows,quarantined_rows
2964624,2712210,252414


In [0]:
display(spark.sql(f"""
SELECT reason, count(*) AS rows_rejected
FROM (SELECT explode(dq_failures) AS reason FROM {T_SILVER_QUARANTINE})
GROUP BY reason ORDER BY rows_rejected DESC
"""))

reason,rows_rejected
invalid_passenger_count,171627
distance_too_short,69739
invalid_fare,37448
non_positive_total,35920
duration_too_short,35121
duration_too_long,1603
implausible_speed,1067
dropoff_before_pickup,870
too_many_passengers,60
distance_too_long,59


In [0]:
display(spark.table(T_SILVER_TRIPS).limit(10))

trip_key,vendor_id,pickup_ts,dropoff_ts,pickup_date,pickup_hour,passenger_count,trip_distance,trip_duration_min,avg_speed_mph,rate_code_id,store_and_fwd_flag,pu_location_id,do_location_id,payment_type_id,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,congestion_surcharge,airport_fee,total_amount,tip_pct,is_airport_trip,_source_file,_ingested_at,_processed_at
0000736f44ae17ce2161cc04690931a00a33d5c519752d18b9c1a85ecffd753a,1,2024-01-19T02:40:33.000Z,2024-01-19T02:47:05.000Z,2024-01-19,2,1,0.9,6.53,8.27,1,N,4,137,1,7.2,3.5,0.5,2.45,0.0,1.0,2.5,0.0,14.65,34.03,false,/Volumes/nyc_taxi/landing/files/raw_trips/yellow_tripdata_2024-01.parquet,2026-10-15T08:45:05.768Z,2026-10-15T08:57:41.079Z
0000c8f2b3580b6fae626f5152602a4d9cf4e01961044e52670aa362bf2b93a0,1,2024-01-13T19:08:47.000Z,2024-01-13T19:26:29.000Z,2024-01-13,19,1,3.5,17.7,11.86,99,N,196,28,1,23.5,0.0,0.5,0.0,0.0,1.0,0.0,0.0,25.0,0.0,false,/Volumes/nyc_taxi/landing/files/raw_trips/yellow_tripdata_2024-01.parquet,2026-10-15T08:45:05.768Z,2026-10-15T08:57:41.079Z
00010fe47d0914ccd7bd22f4057fbf2ce442b583aff18b99db91e349e5073065,2,2024-01-10T15:35:26.000Z,2024-01-10T15:41:31.000Z,2024-01-10,15,1,0.49,6.08,4.84,1,N,143,48,1,7.2,0.0,0.5,1.0,0.0,1.0,2.5,0.0,12.2,13.89,false,/Volumes/nyc_taxi/landing/files/raw_trips/yellow_tripdata_2024-01.parquet,2026-10-15T08:45:05.768Z,2026-10-15T08:57:41.079Z
00011428e62369f8ad726107a69227124479eb31c5960b1ca6f1e0c5f38fbe54,2,2024-01-18T20:56:21.000Z,2024-01-18T21:08:00.000Z,2024-01-18,20,1,1.24,11.65,6.39,1,N,161,50,1,11.4,1.0,0.5,1.5,0.0,1.0,2.5,0.0,17.9,13.16,false,/Volumes/nyc_taxi/landing/files/raw_trips/yellow_tripdata_2024-01.parquet,2026-10-15T08:45:05.768Z,2026-10-15T08:57:41.079Z
000123c9d8da1e024f61007956afacaccd30d136367292064b0b0e88fc46c5cd,2,2024-01-26T15:03:36.000Z,2024-01-26T15:09:03.000Z,2024-01-26,15,1,0.41,5.45,4.51,1,N,249,158,1,6.5,0.0,0.5,2.62,0.0,1.0,2.5,0.0,13.12,40.31,false,/Volumes/nyc_taxi/landing/files/raw_trips/yellow_tripdata_2024-01.parquet,2026-10-15T08:45:05.768Z,2026-10-15T08:57:41.079Z
00013b1891603a301e315f003a02c3db06afe8867a739fb63ad81862d430bdea,2,2024-01-22T18:09:16.000Z,2024-01-22T18:19:06.000Z,2024-01-22,18,2,0.83,9.83,5.07,1,N,100,161,4,10.0,2.5,0.5,0.0,0.0,1.0,2.5,0.0,16.5,0.0,false,/Volumes/nyc_taxi/landing/files/raw_trips/yellow_tripdata_2024-01.parquet,2026-10-15T08:45:05.768Z,2026-10-15T08:57:41.079Z
0001c304a8b6262948bd1e0da82523ad9216907156a1905552bbe7f2b11976b1,2,2024-01-10T18:46:04.000Z,2024-01-10T18:48:49.000Z,2024-01-10,18,1,0.59,2.75,12.87,1,N,186,234,1,5.1,2.5,0.5,2.4,0.0,1.0,2.5,0.0,14.0,47.06,false,/Volumes/nyc_taxi/landing/files/raw_trips/yellow_tripdata_2024-01.parquet,2026-10-15T08:45:05.768Z,2026-10-15T08:57:41.079Z
0002044c3181e90d049953235a7bfa4bda9be3b78c09485b41b4ea9517c586b9,2,2024-01-03T21:28:46.000Z,2024-01-03T21:33:04.000Z,2024-01-03,21,1,0.67,4.3,9.35,1,N,90,68,2,6.5,1.0,0.5,0.0,0.0,1.0,2.5,0.0,11.5,0.0,false,/Volumes/nyc_taxi/landing/files/raw_trips/yellow_tripdata_2024-01.parquet,2026-10-15T08:45:05.768Z,2026-10-15T08:57:41.079Z
00023aed3212ff52644e2c60e9224fa72e1afaccf22b69fb159ead3f0a4a0bae,2,2024-01-17T19:16:15.000Z,2024-01-17T19:20:22.000Z,2024-01-17,19,1,0.75,4.12,10.92,1,N,161,163,1,6.5,2.5,0.5,2.6,0.0,1.0,2.5,0.0,15.6,40.0,false,/Volumes/nyc_taxi/landing/files/raw_trips/yellow_tripdata_2024-01.parquet,2026-10-15T08:45:05.768Z,2026-10-15T08:57:41.079Z
0002596a485dfdf98f6944c60f351c2c3a9b3d1de38ce4176d8ebed4778c72f5,2,2024-01-27T15:04:09.000Z,2024-01-27T15:30:00.000Z,2024-01-27,15,1,3.5,25.85,8.12,1,N,114,163,1,24.0,0.0,0.5,7.0,0.0,1.0,2.5,0.0,35.0,29.17,false,/Volumes/nyc_taxi/landing/files/raw_trips/yellow_tripdata_2024-01.parquet,2026-10-15T08:45:05.768Z,2026-10-15T08:57:41.079Z
